# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Classical machine learning often viewed generative modeling through normalized probabilities. LeCun et al. (2006) formulated **Energy-Based Models (EBMs)** as unnormalized scalar energy landscapes. Smolensky (1986) and Hinton (2002) introduced **Restricted Boltzmann Machines (RBMs)** and Contrastive Divergence, enabling closed-form free energy marginalization over bipartite graphs. Krotov & Hopfield (2016) and Ramsauer et al. (2020) revolutionized associative memory with **Modern Continuous Hopfield Networks**, discovering that modern Hopfield energy minimization directly yields the **Transformer Attention Mechanism**.

### Purpose & Target Architecture
This notebook constructs from first principles:
1. RBM free energy computation marginalizing binary hidden units.
2. Stochastic Gradient Langevin Dynamics (SGLD) sampling on energy surfaces.
3. Continuous modern Hopfield network energy with exponential storage capacity.
4. Proving mathematical equivalence between modern Hopfield retrieval and Dot-Product Attention.

# 2. Mathematical Foundations & Governing Equations

### Boltzmann Distribution & RBM Free Energy
Probability is governed by the Boltzmann distribution:
$$p(v) = \frac{1}{Z} \sum_h e^{-E(v, h)} = \frac{e^{-\mathcal{F}(v)}}{Z}$$
where marginalizing hidden units $h \in \{0, 1\}^H$ yields:
$$\mathcal{F}(v) = -a^T v - \sum_{j=1}^H \text{softplus}(W_{\cdot j}^T v + b_j)$$

### Overdamped Langevin Dynamics
Samples are generated via continuous-time Brownian drift:
$$x_{k+1} = x_k - \frac{\epsilon}{2} \nabla_x E(x_k) + \sqrt{\epsilon} z_k, \quad z_k \sim \mathcal{N}(0, I)$$

### Modern Hopfield Energy and Attention Equivalence
For memory pattern matrix $M \in \mathbb{R}^{M \times D}$, modern continuous Hopfield energy is:
$$E(x) = -\frac{1}{\beta} \text{lse}(\beta, x M^T) + \frac{1}{2} \|x\|^2$$
The associative retrieval fixed point satisfies:
$$x^{\text{new}} = \text{softmax}(\beta \cdot x M^T) M$$
which is the exact mathematical definition of Dot-Product Attention!

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Memory patterns: 4 stored vectors of dimension 8
M, D = 4, 8
memories = torch.randn(M, D)
# Normalize memories to unit sphere for stable associative retrieval
memories = F.normalize(memories, p=2, dim=-1)

# Corrupted query: memory 0 + Gaussian noise
clean_pattern = memories[0].clone()
noisy_query = clean_pattern + torch.randn_like(clean_pattern) * 0.4
noisy_query = noisy_query.unsqueeze(0)  # Shape [1, D]

print(f"Setup complete: memories shape={memories.shape}, query shape={noisy_query.shape}")
print(f"Initial cosine similarity to target: {F.cosine_similarity(noisy_query, clean_pattern.unsqueeze(0)).item():.4f}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def rbm_free_energy(v: torch.Tensor, w: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    v_bias = (v @ a).unsqueeze(-1)
    h_act = F.softplus(v @ w + b)
    return -v_bias - torch.sum(h_act, dim=-1, keepdim=True)

def langevin_dynamics_step(x: torch.Tensor, grad_energy: torch.Tensor, step_size: float, noise: torch.Tensor) -> torch.Tensor:
    return x - 0.5 * step_size * grad_energy + math.sqrt(step_size) * noise

def modern_hopfield_energy(x: torch.Tensor, memories: torch.Tensor, beta: float) -> torch.Tensor:
    sim = x @ memories.T
    lse = torch.logsumexp(beta * sim, dim=-1, keepdim=True) / beta
    norm_term = 0.5 * torch.sum(x ** 2, dim=-1, keepdim=True)
    return -lse + norm_term

def modern_hopfield_retrieve(query: torch.Tensor, memories: torch.Tensor, beta: float) -> torch.Tensor:
    scores = query @ memories.T
    weights = F.softmax(beta * scores, dim=-1)
    return weights @ memories

# Execute one associative memory recall step
beta = 5.0
retrieved = modern_hopfield_retrieve(noisy_query, memories, beta=beta)
post_sim = F.cosine_similarity(retrieved, clean_pattern.unsqueeze(0)).item()
print(f"Hopfield recall complete: recovered cosine similarity = {post_sim:.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: Attention equivalence: modern_hopfield_retrieve matches scaled dot-product attention
q = noisy_query.unsqueeze(1)       # [1, 1, D]
k = memories.unsqueeze(0)          # [1, M, D]
v = memories.unsqueeze(0)          # [1, M, D]
attn_out = F.scaled_dot_product_attention(q, k, v, scale=beta).squeeze(1)
diff = torch.max(torch.abs(retrieved - attn_out)).item()
assert diff < 1e-6, f"Hopfield retrieval must match Attention: error {diff:.2e}"

# Invariant 2: Energy monotonically decreases during CCCP Hopfield update
e_before = modern_hopfield_energy(noisy_query, memories, beta=beta).item()
e_after = modern_hopfield_energy(retrieved, memories, beta=beta).item()
assert e_after <= e_before, f"Hopfield retrieval must decrease energy: before={e_before:.4f}, after={e_after:.4f}"

# Invariant 3: RBM free energy exactness
V_dim, H_dim = 6, 8
v_test = torch.randn(2, V_dim)
w_test = torch.randn(V_dim, H_dim)
a_test = torch.randn(V_dim)
b_test = torch.randn(H_dim)
fe = rbm_free_energy(v_test, w_test, a_test, b_test)
assert fe.shape == (2, 1), f"Free energy shape mismatch: {fe.shape}"
print("All EBM and Hopfield mathematical invariants verified successfully!")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark Hopfield attention retrieval vs Langevin 50-step diffusion
start = time.perf_counter()
for _ in range(500):
    _ = modern_hopfield_retrieve(noisy_query, memories, beta=beta)
hopfield_latency = (time.perf_counter() - start) / 500.0

start = time.perf_counter()
x_mcmc = noisy_query.clone()
for _ in range(50):
    dummy_grad = torch.randn_like(x_mcmc) * 0.1
    dummy_noise = torch.randn_like(x_mcmc)
    x_mcmc = langevin_dynamics_step(x_mcmc, dummy_grad, 0.01, dummy_noise)
langevin_latency = time.perf_counter() - start

print(f"Modern Hopfield single-step recall: {hopfield_latency*1e6:.1f} µs")
print(f"Langevin 50-step MCMC sampling: {langevin_latency*1e3:.2f} ms")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Overly aggressive Langevin step size without noise scaling
def unstable_langevin_bad(x: torch.Tensor, grad_energy: torch.Tensor, step_size: float):
    # Missing 0.5 factor and sqrt(step_size) scaling causes exponential explosion
    return x - step_size * grad_energy + step_size * torch.randn_like(x)

x_bad = noisy_query.clone()
x_good = noisy_query.clone()
large_grad = torch.randn_like(x_bad) * 10.0

# With step_size = 1.0, bad diverges
x_bad_next = unstable_langevin_bad(x_bad, large_grad, step_size=1.0)
x_good_next = langevin_dynamics_step(x_good, large_grad, step_size=0.01, noise=torch.randn_like(x_good))

print(f"Pathological Langevin divergence norm: {torch.norm(x_bad_next).item():.2f} (Exploding state)")
print(f"Corrected SGLD stable norm: {torch.norm(x_good_next).item():.2f} (Well-conditioned)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Architectural Insights
1. **Unification of Hopfield and Transformers**: Dot-product attention is an energy-minimization retrieval step on a continuous modern Hopfield landscape with exponential memory storage capacity.
2. **Energy vs Likelihood**: EBMs decouple representation from normalization, enabling compositionality ($E_{A \land B}(x) = E_A(x) + E_B(x)$) without intractable partition function computation.
3. **Langevin Dynamics as Continuous Sampling**: Overdamped Langevin dynamics bridges continuous score-based diffusion models and classical statistical mechanics.